In [1]:
import happybase

# Connect to the running HBase Thrift server
connection = happybase.Connection('localhost', port=9090)
connection.open()
table = connection.table('user_browsing_data')

# We pick 'user_0' as our target since your generator starts with sequential IDs
target_user = "user_0" 

print(f"=== Retrieving Session Activity Logs for: {target_user} (Latest First) ===\n")

records_found = 0

# Prefix scan quickly targets ONLY the rows belonging to user_0
for key, data in table.scan(row_prefix=target_user.encode()):
    records_found += 1
    
    # Clean up the output decoding byte strings to standard text
    row_key = key.decode()
    session_id = data.get(b'cf_session:session_id', b'unknown').decode()
    duration = data.get(b'cf_session:duration', b'0').decode()
    device = data.get(b'cf_session:device', b'unknown').decode()
    products = data.get(b'cf_session:products', b'').decode()
    
    print(f"[Row Key]: {row_key}")
    print(f"  Session ID: {session_id}")
    print(f"  Duration:   {duration} seconds")
    print(f"  Device:     {device}")
    print(f"  Products:   {products}")
    print("-" * 50)
    
    # Stop printing after 5 records so your terminal doesn't get flooded
    if records_found >= 5:
        print("... showing the top 5 most recent sessions ...")
        break

if records_found == 0:
    print(f"No records found for {target_user}. Try checking 'user_1' or 'user_2'.")

connection.close()


=== Retrieving Session Activity Logs for: user_0 (Latest First) ===

[Row Key]: user_000000_9223372035073354053
  Session ID: sess_fd30e61a5d
  Duration:   2795 seconds
  Device:     desktop
  Products:   prod_00369,prod_00117,prod_00313,prod_01184
--------------------------------------------------
[Row Key]: user_000000_9223372035073990951
  Session ID: sess_569641ff5b
  Duration:   1423 seconds
  Device:     desktop
  Products:   prod_03465,prod_03177,prod_02729,prod_04062,prod_01955,prod_01364,prod_02947
--------------------------------------------------
[Row Key]: user_000000_9223372035075132828
  Session ID: sess_d2723f84b1
  Duration:   100 seconds
  Device:     desktop
  Products:   prod_02514,prod_03068,prod_01665
--------------------------------------------------
[Row Key]: user_000000_9223372035075544069
  Session ID: sess_3216efcfee
  Duration:   1287 seconds
  Device:     tablet
  Products:   prod_01004,prod_00578,prod_02666,prod_03220,prod_02423
---------------------------